In [ ]:
pip install speechrecognition

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.8/32.8 MB 36.6 MB/s eta 0:00:00


In [ ]:
!sudo apt-get install python3-pyaudio

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libportaudio2
Suggested packages:
  python-pyaudio-doc
The following NEW packages will be installed:
  libportaudio2 python3-pyaudio
0 upgraded, 2 newly installed, 0 to remove and 49 not upgraded.
Need to get 91.2 kB of archives.
After this operation, 340 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libportaudio2 amd64 19.6.0-1.1 [65.3 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 python3-pyaudio amd64 0.2.11-1.3ubuntu1 [25.9 kB]
Fetched 91.2 kB in 1s (79.8 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 2.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readlin

In [ ]:
pip install pyaudio

In [ ]:
pip install pyttsx3

In [ ]:
def init(driverName, debug):
    pass

In [ ]:
def say(text, name):
    pass

In [ ]:
import speech_recognition as sr
import pyttsx3

# Initialize the recognizer
r = sr.Recognizer()

# Function to convert text to speech
def SpeakText(command):
    # Initialize the engine
    engine = pyttsx3.init()
    engine.say(command)
    engine.runAndWait()

# Loop infinitely for user to speak
while True:
    try:
        # Use the microphone as source for input
        with sr.Microphone() as source2:
            # Adjust the recognizer sensitivity to ambient noise
            r.adjust_for_ambient_noise(source2, duration=0.2)

            # Listen for the user's input
            print("Listening...")
            audio2 = r.listen(source2)

            # Using Google to recognize audio
            MyText = r.recognize_google(audio2)
            MyText = MyText.lower()

            print("Did you say: ", MyText)
            SpeakText(MyText)

    except sr.RequestError as e:
        print(f"Could not request results; {e}")

    except sr.UnknownValueError:
        print("Unknown error occurred")


OSError: No Default Input Device Available

In [ ]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 25.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from IPython import display
from jiwer import wer

In [ ]:
data_url = "https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2"

In [ ]:
data_path = keras.utils.get_file("LJSpeech-1.1" , data_url , untar = True)

2748572632/2748572632 ━━━━━━━━━━━━━━━━━━━━ 59s 0us/step


In [ ]:
wavs_path = data_path + "/wavs/"
metadata_path = data_path + "/metadata.csv"

In [ ]:
metadata_df = pd.read_csv(metadata_path , sep = "|" , header = None , quoting = 3)

In [ ]:
metadata_df.tail()

,0,1,2
13095,LJ050-0274,made certain recommendations which it believes...,made certain recommendations which it believes...
13096,LJ050-0275,materially improve upon the procedures in effe...,materially improve upon the procedures in effe...
13097,LJ050-0276,"As has been pointed out, the Commission has no...","As has been pointed out, the Commission has no..."
13098,LJ050-0277,with the active cooperation of the responsible...,with the active cooperation of the responsible...
13099,LJ050-0278,the recommendations we have here suggested wou...,the recommendations we have here suggested wou...


In [ ]:
metadata_df.columns = ["file_name" , "transcription" , "normalized_transcription"]
metadata_df = metadata_df[["file_name" , "normalized_transcription"]]
metadata_df = metadata_df.sample(frac = 1).reset_index(drop = True)
metadata_df.head(5)

NameError: name 'metadata_df' is not defined

In [ ]:
metadata_df.columns = ["file_name" , "transcription" , "normalized_transcription"]
metadata_df = metadata_df[["file_name" , "transcription"]]
metadata_df = metadata_df.sample(frac = 1).reset_index(drop = True)
metadata_df.head()

NameError: name 'metadata_df' is not defined

In [ ]:
split = int(len(metadata_df)* 0.90)
df_train = metadata_df[:split]
df_val = metadata_df[split:]
print(f"Size of the training set: {len(df_train)}")
print(f"Size of the training set: {len(df_val)}")

In [ ]:
characters = [x for x in "abcdefghijklmnopqrstuvwxyz' "]
char_to_num = keras.layers.StringLookup(vocabulary = characters , oov_token = "")
num_to_char = keras.layers.StringLookup(vocabulary = char_to_num.get_vocabulary() , oov_token = "" , invert = True)
print(
    f"The vocabulary = {char_to_num.get_vocabulary()} "
    f"(size = {char_to_num.vocabulary_size()})"
)

In [ ]:
char_to_num

In [ ]:
frame_length = 256
frame_step = 160
fft_length = 384
def encode_single_sample(wav_file , label):
    file = tf.io.read_file(wavs_path + wav_file + ".wav")
    audio , _ = tf.audio.decode_wav(file)
    audio = tf.squeeze(audio , axis = -1)
    audio = tf.cast(audio , tf.float32)
    spectrogram = tf.signal.stft(audio , frame_length = frame_length , frame_step = frame_step , fft_length = fft_length)
    spectrogram = tf.abs(spectrogram)
    spectrogram = tf.math.pow(spectrogram , 0.5)
    means = tf.math.reduce_mean(spectrogram , 1 , keepdims = True)
    stddevs = tf.math.reduce_std(spectrogram , 1 , keepdims = True)
    spectrogram = (spectrogram - means) / (stddevs + 1e-10)
    label = tf.strings.lower(label)
    label = tf.strings.unicode_split(label , input_encoding = "UTF-8")
    label = char_to_num(label)
    return spectrogram , label


In [ ]:
batch_size = 32
train_dataset = tf.data.Dataset.from_tensor_slices((list(df_train["file_name"]) , list(df_train["transcription"])))
train_dataset = (
    train_dataset.map(encode_single_sample , num_parallel_calls = tf.data.AUTOTUNE)
    .padded_batch(batch_size)
    .prefetch(buffer_size = tf.data.AUTOTUNE)
)
val_dataset = tf.data.Dataset.from_tensor_slices((list(df_val["file_name"]) , list(df_val["transcription"])))
val_dataset = (
    val_dataset.map(encode_single_sample , num_parallel_calls = tf.data.AUTOTUNE)
    .padded_batch(batch_size)
    .prefetch(buffer_size = tf.data.AUTOTUNE)
)


In [ ]:
fig = plt.figure(figsize = (8 , 5))
for batch in train_dataset.take(1):
    spectrogram = batch[0][0].numpy()
    spectrogram = np.array([np.trim_zeros(x) for x in np.transpose(spectrogram)])
    label = batch[1][0]
    label = tf.strings.reduce_join(num_to_char(label)).numpy().decode("UTF-8")
    ax = plt.subplot(2 , 1 , 1)
    ax.imshow(spectrogram , cmap = "tinysrgb" , aspect = "auto")
    ax.set_title(f"Label: {label}")
    ax.set_xlabel("Time axis")
    ax.set_ylabel("Frequency axis")
    ax.set_colorbar(None)
    ax.axis("off")
    file = tf.io.read_file(wavs_path + batch[0][0].numpy()
    audio , _ = tf.audio.decode_wav(file)
    ax = plt.subplot(2 , 1 , 2)
    ax.plot(audio.numpy())
    ax.set_title("Waveform")
    ax.set_xlabel("Time axis")
    ax.set_ylabel("Amplitude")
    display.display(display.Audio(audio , rate = 16000))
plt.show()

In [ ]:
def CTCLoss(y_true , y_pred):
    batch_len = tf.cast(tf.shape(y_true)[0] , dtype = "int64")
    input_length = tf.cast(tf.shape(y_pred)[1] , dtype = "int64")
    label_length = tf.cast(tf.shape(y_true)[1] , dtype = "int64")
    input_length = input_length * tf.ones(shape = (batch_len , 1) , dtype = "int64")
    label_length = label_length * tf.ones(shape = (batch_len , 1) , dtype = "int64")
    loss = keras.backend.ctc_batch_cost(y_true , y_pred , input_length , label_length)
    return loss


In [ ]:
def build_model(input_dim , output_dim , rnn_layers = 5 , rnn_units = 128):
    input_spectrogram = layers.Input((None , input_dim) , name = "input")
    x = layers.Reshape((-1 , input_dim , 1) , name = "expand_dim")(input_spectrogram)
    x = layers.Conv2D(
        filters = 32,
        kernel_size = [11 , 41],
        strides = [2 , 2],
        padding = "same",
        use_bias = False
        name = "conv_1",
    )(x)
    x = layers.BatchNormalization(name = "conv_1_bn")(x)
    x = layers.ReLU(name = "conv_1_relu")(x)
    x = layers.Conv2D(
        filters = 32,
        kernel_size = [11 , 21],
        strides = [1 , 2],
        padding = "same",
        use_bias = False,
        name = "conv_2",
    )(x)
    x = layers.BatchNormalization(name = "conv_2_bn")(x)
    x = layers.ReLU(name = "conv_2_relu")(x)
    x = layers.Reshape((-1 , x.shape[-2] * x.shape[-1]))(x)
    for i in range(1 , rnn_layers + 1):
      recurrent = layers.GRU(
          units = rnn_units,
          activation = "tanh",
          recurrent_activation = "sigmoid",
          use_bias = True,
          return_sequences = True,
          reset_after = True,
          name = f"gru_{i}",
      )
      x = layers.Bidirectional(
          recurent, name = f"bidirectional_{i}", merge_mode = "concat"
      )(x)
      if i < rnn_layers:
        x = layers.Dropout(rate = 0.5)(x)
    x = layers.Dense(units = rnn_units * 2 , name = "dense_1")(x)
    x = layers.ReLU(name = "dense_1_relu")(x)
    x = layers.Dropout(rate = 0.5)(x)
    output = layers.Dense(units = output_dim + 1 , activation = "softmax")(x)
    model = keras.Model(input_spectrogram , output , name = "DeepSpeech_2")
    opt = keras.optimizers.Adam(learning_rate = 1e-4)
    model.compile(optimizer = opt , loss = CTCLoss)
    return model
    model = build_model(
        input_dim = fft length // 2 + 1,
        output_dim = char_to_num.vocabulary_size(),
        rnn_units = 512,
        rnn_layers = 5,
    )
    model.summary(line_length = 110)

In [ ]:
def decode_batch_predictions(pred):
    input_len = np.ones(pred.shape[0]) * pred.shape[1]
    results = keras.backend.ctc_decode(pred , input_length = input_length = input_len , greedy=True)[0][0]
    output_text = []
    for result in results:
      result = tf.strings.reduce_join(num_to_char(result)).numpy().decode("utf-8")
      output_text.append(result)
    return output_text